In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.gold;

In [0]:
%sql
-- Materialized table for AI export - refreshed daily by pipeline
CREATE OR REPLACE TABLE garmin_lakehouse.gold.running_splits AS
SELECT
    s.activity_id,
    s.lap_index,
    a.start_date,                          -- from the activity, for time-based filtering
    s.start_time_gmt,

    s.intensity_type,
    CASE s.intensity_type
        WHEN 'ACTIVE'   THEN 'work'
        WHEN 'RECOVERY' THEN 'recovery'
        WHEN 'WARMUP'   THEN 'warmup'
        WHEN 'COOLDOWN' THEN 'cooldown'
        ELSE 'unstructured'                -- null = auto-lapped, no programmed structure
    END AS lap_role,

    s.distance_km,
    s.duration_minutes,
    s.moving_duration_minutes,
    s.avg_pace_min_per_km,
    s.avg_moving_pace_min_per_km,
    s.best_pace_min_per_km,

    s.avg_heart_rate,
    s.max_heart_rate,
    s.avg_cadence_rpm,
    s.stride_length_m,

    s.elevation_gain_m,
    s.elevation_loss_m,
    s.calories,
    s.loaded_at
FROM garmin_lakehouse.silver.splits s
INNER JOIN garmin_lakehouse.gold.running_activities a
    ON s.activity_id = a.activity_id

In [0]:
%sql
SELECT COUNT(DISTINCT activity_id) from garmin_lakehouse.gold.running_splits